# SIHEDAF 2.0 — Revisi Implementasi (v2)

Notebook ini merevisi pipeline lama (`ekstraksi_hrv.py`, `model.ipynb`, `predictor.py`, `final.joblib`) berdasarkan hasil analisis repo.

| # | Temuan pada versi lama | Perbaikan di notebook ini |
|---|---|---|
| 1 | **Training–serving skew**: latih pakai R-peak anotasi 250 Hz, produksi pakai `find_peaks` mentah pada 50 Hz; filter outlier 300–2000 ms tidak dipakai saat inferensi; batas 150 bpm | Modul bersama `hrv_core.py` dipakai ekstraksi, training, dan predictor. Detektor R-peak ala Pan-Tompkins yang jalan di 50 Hz (batas 240 bpm, interpolasi sub-sampel). Dataset kedua diekstrak **dengan detektor 50 Hz**, lalu diuji eksperimen pergeseran domain |
| 2 | Sel inferensi diam-diam memakai bobot acak + scaler dummy; IBI dalam detik | Predictor baru tanpa fallback diam-diam: file hilang → error, sinyal buruk → status `tidak_terbaca`, IBI dalam detik → error |
| 3 | Asal-usul `final.joblib` tidak bisa ditelusuri; path default salah; pickle | Training final + ekspor di notebook ini. Bobot `.pt` (state_dict, `weights_only=True`) + metadata `.json` + checksum SHA-256 + konstanta praproses |
| 4 | Evaluasi AFL rapuh: AFL hanya di sedikit pasien; stride AFL 1 s (overlap 90%) | Audit pasien AFL per fold, stride seragam (default 5 s), metrik **pooled out-of-fold**, recall AFL per pasien, kalibrasi bias kelas dengan *cross-fitting* |
| 5 | Perbandingan ML vs DL tidak adil (6 fitur lemah, latensi GPU vs CPU) | 16 fitur HRV (RMSSD, pNN50, SD1/SD2, TPR, SampEn, COSEn, …), tanpa scaler untuk pohon, latensi diukur di CPU dengan `perf_counter` termasuk praproses |
| 6 | 1D-ResNet mengabaikan mask | `MaskedResNet1D` + uji invariansi padding. Ditambah `BiGRUAttention` sebagai pembanding |

**Cara pakai.** Taruh notebook ini di root repo. Data AFDB diharapkan di folder `files/` (sel unduh tersedia). Jalankan berurutan dari atas. Hasil ekstraksi di-cache di `cache/`, model akhir di `model_export/`.

> Untuk uji cepat alur notebook (bukan untuk hasil), set environment variable `SIHEDAF_QUICK=1` sebelum menjalankan: epoch dikurangi menjadi 2.

## 0. Setup & konfigurasi

In [ ]:
import os, sys, json, time, random, hashlib, platform, subprocess, importlib, warnings
from dataclasses import dataclass, field, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

import lightgbm as lgb
import wfdb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

QUICK = os.environ.get("SIHEDAF_QUICK") == "1"


@dataclass
class Config:
    # ── path ──
    data_dir: str = "files"            # folder AFDB (RECORDS, .dat, .hea, .atr, .qrs/.qrsc)
    cache_dir: str = "cache"
    out_dir: str = "model_export"
    blacklist: tuple = ("00735", "03665")   # rekaman AFDB tanpa file sinyal
    ecg_channel: int = 0               # lead yang disimulasikan sebagai lead perangkat

    # ── windowing ──
    # Versi lama: {0: 5.0, 1: 10.0, 2: 1.0} → window AFL tumpang-tindih 90%,
    # jumlah sampel AFL terlihat besar padahal informasinya sedikit.
    stride_sec: dict = field(default_factory=lambda: {0: 5.0, 1: 5.0, 2: 5.0})
    invalid_step_sec: float = 1.0      # geser 1 s jika window tidak valid

    # ── CV & training ──
    n_splits: int = 5
    seed: int = 42
    epochs: int = 30
    batch_size: int = 256
    lr: float = 1e-3
    weight_decay: float = 1e-4
    dl_models: tuple = ("MicroTransformer", "MaskedResNet1D", "BiGRUAttention")

    # ── model akhir ──
    final_model: str = "MicroTransformer"
    final_source: str = "detector"     # "detector" = sama dengan kondisi produksi 50 Hz

    # ── uji data perangkat ──
    raw_test_file: str = "raw_71.json"
    raw_test_fs: int = 50


cfg = Config()
if QUICK:
    cfg.epochs = 2

MODEL_KWARGS = {
    "MicroTransformer": dict(input_dim=2, d_model=32, nhead=2, dim_ff=64, num_classes=3, dropout=0.1),
    "MaskedResNet1D":   dict(input_dim=2, num_classes=3),
    "BiGRUAttention":   dict(input_dim=2, hidden=32, num_classes=3),
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


for d in (cfg.cache_dir, cfg.out_dir):
    Path(d).mkdir(exist_ok=True)

set_seed(cfg.seed)
print(f"Device: {DEVICE} | torch {torch.__version__} | QUICK={QUICK}")
print(asdict(cfg))

## 1. Modul bersama

Dua modul ditulis ke disk agar **notebook dan backend memakai kode yang sama**:

- `hrv_core.py` — konstanta praproses, detektor R-peak, IBI, aturan kualitas window, pembentukan sekuens, fitur HRV.
- `sihedaf_models.py` — arsitektur model.

Backend cukup menyalin `hrv_core.py`, `sihedaf_models.py`, `predictor.py`, dan folder `model_export/`.

In [ ]:
%%writefile hrv_core.py
"""
hrv_core.py — SATU sumber kebenaran untuk praproses SIHEDAF.

Dipakai oleh ekstraksi dataset, training, DAN predictor backend, sehingga
data latih dan data produksi diproses dengan kode yang persis sama
(menutup masalah training-serving skew pada versi sebelumnya).

Konvensi:
- IBI selalu dalam MILIDETIK.
- Sekuens model berbentuk (MAX_BEATS, 2) = [IBI, delta_IBI], delta_IBI[0] = 0.
- Mask: 1 = detak valid, 0 = padding.
"""
from fractions import Fraction

import numpy as np
from scipy import signal as sps

# ── Konstanta praproses (ikut diekspor ke metadata model) ──────────────
FS_DEVICE = 50          # sampling rate perangkat produksi (Hz)
WINDOW_SEC = 10.0       # panjang window analisis (detik)
MAX_BEATS = 30          # panjang sekuens model
IBI_MIN_MS = 300.0      # batas fisiologis bawah (200 bpm)
IBI_MAX_MS = 2000.0     # batas fisiologis atas (30 bpm)
MAX_OUTLIERS = 1        # maksimum IBI di luar rentang per window
MIN_IBI = 4             # minimum IBI per window agar window dianggap valid

LABELS_MAP = {0: "Normal", 1: "AFIB (Atrial Fibrillation)", 2: "AFL (Atrial Flutter)"}

FEATURE_NAMES = [
    "mean_ibi", "median_ibi", "min_ibi", "max_ibi", "sdnn", "rmssd",
    "pnn50", "cv", "sd1", "sd2", "sd1_sd2", "tpr", "sampen", "cosen",
    "hr_mean", "beat_count",
]


def preprocessing_config() -> dict:
    """Konstanta praproses, untuk disimpan di metadata model."""
    return {
        "fs_device": FS_DEVICE, "window_sec": WINDOW_SEC, "max_beats": MAX_BEATS,
        "ibi_min_ms": IBI_MIN_MS, "ibi_max_ms": IBI_MAX_MS,
        "max_outliers": MAX_OUTLIERS, "min_ibi": MIN_IBI,
        "ibi_unit": "ms", "sequence_features": ["ibi_ms", "delta_ibi_ms"],
    }


# ── Sinyal → R-peak ────────────────────────────────────────────────────
def resample_signal(x: np.ndarray, fs_in: float, fs_out: float = FS_DEVICE) -> np.ndarray:
    """Resample polifase (anti-aliasing bawaan)."""
    if fs_in == fs_out:
        return np.asarray(x, dtype=np.float64)
    frac = Fraction(int(round(fs_out)), int(round(fs_in))).limit_denominator(1000)
    return sps.resample_poly(np.asarray(x, dtype=np.float64), frac.numerator, frac.denominator)


def detect_rpeaks(ecg: np.ndarray, fs: float, chunk_sec: float = 10.0) -> np.ndarray:
    """
    Detektor R-peak ala Pan-Tompkins yang tahan untuk fs rendah (50 Hz).

    1. Clip artefak ekstrem, bandpass 5–15 Hz.
    2. Turunan → kuadrat → moving-window integration (150 ms).
    3. Threshold adaptif per potongan `chunk_sec` detik.
    4. Jarak minimum 250 ms (sampai 240 bpm; versi lama dibatasi 150 bpm,
       padahal AFL 2:1 sering tepat di ~150 bpm).
    5. Refinement ke puncak |sinyal| + interpolasi parabolik sub-sampel,
       untuk mengurangi kuantisasi 20 ms pada 50 Hz.

    Return: posisi R-peak dalam satuan SAMPEL (float, boleh pecahan).
    """
    x = np.nan_to_num(np.asarray(ecg, dtype=np.float64).ravel())
    if x.size < int(2 * fs):
        return np.array([], dtype=np.float64)

    lo, hi = np.percentile(x, [0.5, 99.5])
    x = np.clip(x, lo, hi)

    nyq = fs / 2.0
    high = min(15.0, 0.8 * nyq)
    b, a = sps.butter(2, [5.0 / nyq, high / nyq], btype="band")
    y = sps.filtfilt(b, a, x)

    d = np.gradient(y)
    w = max(1, int(round(0.15 * fs)))
    mwi = np.convolve(d * d, np.ones(w) / w, mode="same")

    # Threshold adaptif per chunk
    thr = np.empty_like(mwi)
    n_chunk = max(1, int(round(chunk_sec * fs)))
    for s in range(0, mwi.size, n_chunk):
        seg = mwi[s:s + n_chunk]
        thr[s:s + n_chunk] = 0.3 * np.percentile(seg, 98)

    cand, _ = sps.find_peaks(mwi, distance=max(1, int(round(0.25 * fs))))
    cand = cand[mwi[cand] >= thr[cand]]
    if cand.size == 0:
        return np.array([], dtype=np.float64)

    # Refinement ke puncak |y| di sekitar kandidat (±80 ms)
    half = max(1, int(round(0.08 * fs)))
    ay = np.abs(y)
    peaks = []
    for c in cand:
        s, e = max(0, c - half), min(ay.size, c + half + 1)
        p = s + int(np.argmax(ay[s:e]))
        # interpolasi parabolik sub-sampel
        if 0 < p < ay.size - 1:
            y0, y1, y2 = ay[p - 1], ay[p], ay[p + 1]
            den = y0 - 2 * y1 + y2
            off = 0.5 * (y0 - y2) / den if den != 0 else 0.0
            off = float(np.clip(off, -0.5, 0.5))
        else:
            off = 0.0
        peaks.append(p + off)

    peaks = np.unique(np.round(np.asarray(peaks), 3))
    # buang duplikat yang terlalu dekat setelah refinement
    keep = np.concatenate(([True], np.diff(peaks) >= 0.25 * fs))
    return peaks[keep]


# ── R-peak → IBI ───────────────────────────────────────────────────────
def peaks_to_ibi(peaks: np.ndarray, fs: float):
    """Return (t_end_detik, ibi_ms). t_end = waktu R-peak penutup tiap IBI."""
    peaks = np.asarray(peaks, dtype=np.float64)
    if peaks.size < 2:
        return np.array([]), np.array([])
    ibi_ms = np.diff(peaks) / fs * 1000.0
    t_end = peaks[1:] / fs
    return t_end, ibi_ms


def window_quality(ibi_ms: np.ndarray):
    """Return (ok, alasan). Aturan yang SAMA untuk training dan inferensi."""
    ibi_ms = np.asarray(ibi_ms, dtype=np.float64)
    if ibi_ms.size < MIN_IBI:
        return False, f"terlalu sedikit detak ({ibi_ms.size} IBI < {MIN_IBI})"
    n_out = int(np.sum((ibi_ms < IBI_MIN_MS) | (ibi_ms > IBI_MAX_MS)))
    if n_out > MAX_OUTLIERS:
        return False, f"{n_out} IBI di luar {IBI_MIN_MS:.0f}–{IBI_MAX_MS:.0f} ms"
    return True, "ok"


def build_sequence(ibi_ms: np.ndarray):
    """IBI satu window → (X[MAX_BEATS,2], mask[MAX_BEATS]). delta_IBI[0] = 0."""
    ibi = np.asarray(ibi_ms, dtype=np.float32)[:MAX_BEATS]
    n = ibi.size
    X = np.zeros((MAX_BEATS, 2), dtype=np.float32)
    mask = np.zeros(MAX_BEATS, dtype=np.float32)
    X[:n, 0] = ibi
    if n > 1:
        X[1:n, 1] = np.diff(ibi)
    mask[:n] = 1.0
    return X, mask


# ── Fitur HRV untuk cabang tabular ─────────────────────────────────────
def _sample_entropy(x: np.ndarray, m: int, r: float) -> float:
    """SampEn sederhana (O(n²), aman untuk n ≤ ~40)."""
    n = x.size
    if n <= m + 1 or r <= 0:
        return np.nan

    def _count(mm):
        tpl = np.array([x[i:i + mm] for i in range(n - m)])
        dist = np.max(np.abs(tpl[:, None, :] - tpl[None, :, :]), axis=2)
        iu = np.triu_indices(len(tpl), k=1)
        return np.sum(dist[iu] <= r)

    B, A = _count(m), _count(m + 1)
    if A == 0 or B == 0:
        # batas atas yang terdefinisi (hindari inf)
        return float(np.log((n - m) * (n - m - 1) / 2.0))
    return float(-np.log(A / B))


def hrv_features(ibi_ms: np.ndarray) -> np.ndarray:
    """Vektor fitur HRV sesuai urutan FEATURE_NAMES."""
    ibi = np.asarray(ibi_ms, dtype=np.float64)
    d = np.diff(ibi)
    mean = ibi.mean()
    sdnn = ibi.std(ddof=1) if ibi.size > 1 else 0.0
    rmssd = np.sqrt(np.mean(d ** 2)) if d.size else 0.0
    pnn50 = np.mean(np.abs(d) > 50.0) if d.size else 0.0
    sd1 = np.sqrt(0.5) * d.std(ddof=1) if d.size > 1 else 0.0
    sd2_sq = 2 * sdnn ** 2 - sd1 ** 2
    sd2 = np.sqrt(sd2_sq) if sd2_sq > 0 else 0.0
    sd1_sd2 = sd1 / sd2 if sd2 > 0 else 0.0
    if ibi.size >= 3:
        mid = ibi[1:-1]
        tp = np.sum(((mid > ibi[:-2]) & (mid > ibi[2:])) | ((mid < ibi[:-2]) & (mid < ibi[2:])))
        tpr = tp / (ibi.size - 2)
    else:
        tpr = 0.0
    sampen = _sample_entropy(ibi, m=1, r=0.2 * sdnn) if sdnn > 0 else 0.0
    # COSEn (Lake & Moorman): SampEn dengan r tetap + ln(2r) − ln(mean RR), satuan detik
    r_s = 0.03
    se_abs = _sample_entropy(ibi / 1000.0, m=1, r=r_s)
    cosen = se_abs + np.log(2 * r_s) - np.log(mean / 1000.0) if np.isfinite(se_abs) else 0.0
    feats = [
        mean, np.median(ibi), ibi.min(), ibi.max(), sdnn, rmssd, pnn50,
        sdnn / mean if mean > 0 else 0.0, sd1, sd2, sd1_sd2, tpr,
        np.nan_to_num(sampen), np.nan_to_num(cosen), 60000.0 / mean, float(ibi.size),
    ]
    return np.asarray(feats, dtype=np.float32)


# ── Sinyal penuh → window siap model (dipakai predictor) ───────────────
def signal_to_windows(ecg: np.ndarray, fs: float, stride_sec: float = 5.0):
    """
    EKG mentah (fs berapa pun) → list window.
    Sinyal diproses di FS_DEVICE agar identik dengan kondisi produksi.
    """
    x = resample_signal(ecg, fs, FS_DEVICE)
    peaks = detect_rpeaks(x, FS_DEVICE)
    t_end, ibi = peaks_to_ibi(peaks, FS_DEVICE)
    dur = x.size / FS_DEVICE
    out = []
    starts = np.arange(0.0, max(dur - WINDOW_SEC, 0.0) + 1e-9, stride_sec)
    for s in starts:
        i0, i1 = np.searchsorted(t_end, [s, s + WINDOW_SEC], side="left")
        w_ibi = ibi[i0:i1]
        ok, reason = window_quality(w_ibi)
        out.append({"start_sec": float(s), "ibi_ms": w_ibi, "ok": ok, "reason": reason})
    return out, peaks

In [ ]:
%%writefile sihedaf_models.py
"""
sihedaf_models.py — definisi arsitektur, dipakai notebook DAN predictor.
Satu definisi → tidak ada lagi salinan kelas yang bisa berbeda diam-diam.
"""
import numpy as np
import torch
import torch.nn as nn


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 50):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        if d_model > 1:
            pe[:, 1::2] = torch.cos(pos * div[: d_model // 2])
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


def masked_mean(x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """x: (B, S, C), mask: (B, S) → (B, C), rata-rata hanya detak valid."""
    m = mask.unsqueeze(-1)
    return (x * m).sum(1) / m.sum(1).clamp(min=1)


class MicroTransformer(nn.Module):
    """Sama persis dengan versi lama (nama parameter kompatibel)."""
    def __init__(self, input_dim=2, d_model=32, nhead=2, dim_ff=64, num_classes=3, dropout=0.1):
        super().__init__()
        self.embedding = nn.Linear(input_dim, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            batch_first=True, dropout=dropout, activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=1, enable_nested_tensor=False)
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x, mask):
        h = self.pos_enc(self.embedding(x))
        h = self.encoder(h, src_key_padding_mask=(mask == 0))
        return self.fc(masked_mean(h, mask))


class _ResBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.conv1 = nn.Conv1d(cin, cout, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(cout)
        self.conv2 = nn.Conv1d(cout, cout, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(cout)
        self.skip = nn.Conv1d(cin, cout, 1, bias=False) if cin != cout else nn.Identity()
        self.act = nn.SiLU()  # = Swish

    def forward(self, x, m):          # x: (B, C, S), m: (B, 1, S)
        out = self.act(self.bn1(self.conv1(x))) * m
        out = self.bn2(self.conv2(out))
        return self.act(out + self.skip(x)) * m


class MaskedResNet1D(nn.Module):
    """
    1D-ResNet yang MEMAKAI mask (versi lama mengabaikannya sehingga
    posisi padding ikut dirata-rata di global pooling).
    """
    def __init__(self, input_dim=2, num_classes=3, channels=(32, 64, 128)):
        super().__init__()
        chans = (input_dim,) + tuple(channels)
        self.blocks = nn.ModuleList(_ResBlock(chans[i], chans[i + 1]) for i in range(len(channels)))
        self.fc = nn.Linear(chans[-1], num_classes)

    def forward(self, x, mask):
        m = mask.unsqueeze(1)                 # (B, 1, S)
        h = x.permute(0, 2, 1) * m
        for blk in self.blocks:
            h = blk(h, m)
        return self.fc(masked_mean(h.permute(0, 2, 1), mask))


class BiGRUAttention(nn.Module):
    """Bi-GRU + attention pooling bermask. Ringan (~15 ribu parameter)."""
    def __init__(self, input_dim=2, hidden=32, num_classes=3):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden, batch_first=True, bidirectional=True)
        self.att = nn.Linear(2 * hidden, 1)
        self.fc = nn.Linear(2 * hidden, num_classes)

    def forward(self, x, mask):
        lengths = mask.sum(1).clamp(min=1).long().cpu()
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        out, _ = self.gru(packed)
        h, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        score = self.att(h).squeeze(-1).masked_fill(mask == 0, -1e9)
        w = torch.softmax(score, dim=1).unsqueeze(-1)
        return self.fc((h * w).sum(1))


MODEL_REGISTRY = {
    "MicroTransformer": MicroTransformer,
    "MaskedResNet1D": MaskedResNet1D,
    "BiGRUAttention": BiGRUAttention,
}


def build_model(name: str, **kwargs) -> nn.Module:
    return MODEL_REGISTRY[name](**kwargs)


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

In [ ]:
import hrv_core, sihedaf_models
importlib.reload(hrv_core); importlib.reload(sihedaf_models)
from sihedaf_models import build_model, count_params

# ── Uji 1: satuan & delta pertama ──
X, m = hrv_core.build_sequence(np.array([800., 820., 790., 805.]))
assert X[0, 1] == 0 and X[1, 1] == 20 and m.sum() == 4, "build_sequence tidak sesuai konvensi"

# ── Uji 2: semua model invarian terhadap isi padding ──
x = torch.randn(4, hrv_core.MAX_BEATS, 2)
mk = torch.ones(4, hrv_core.MAX_BEATS); mk[:, 12:] = 0
x2 = x.clone(); x2[:, 12:] = torch.randn_like(x2[:, 12:]) * 100   # padding diisi sampah
for name, kw in MODEL_KWARGS.items():
    net = build_model(name, **kw).eval()
    with torch.no_grad():
        diff = (net(x * mk.unsqueeze(-1), mk) - net(x2, mk)).abs().max().item()
    print(f"{name:<18} params={count_params(net):>7,}  selisih output saat padding diubah = {diff:.2e}")
    assert diff < 1e-4, f"{name} tidak menghormati mask"
print("✓ Semua uji modul lolos")

## 2. Data AFDB

In [ ]:
if not (Path(cfg.data_dir) / "RECORDS").exists():
    print("Mengunduh MIT-BIH AFDB dari PhysioNet (± 600 MB, sekali saja)...")
    wfdb.dl_database("afdb", dl_dir=cfg.data_dir)
print("RECORDS:", (Path(cfg.data_dir) / "RECORDS").read_text().split())

## 3. Ekstraksi v2 — dua sumber R-peak

Dataset diekstrak **dua kali** dengan aturan window yang identik:

- `annotation` — R-peak dari anotasi `.qrsc`/`.qrs` pada 250 Hz (seperti versi lama, *gold standard*).
- `detector` — sinyal lead `ecg_channel` di-resample ke **50 Hz**, R-peak dicari dengan `hrv_core.detect_rpeaks`. Ini mensimulasikan kondisi perangkat produksi.

Label tetap dari anotasi ritme `.atr`. Window hanya dipakai jika seluruh detaknya berlabel sama dan lolos `hrv_core.window_quality` (aturan yang sama dipakai predictor). Penentuan IBI dalam window memakai `searchsorted` (versi lama O(n²)).

In [ ]:
class AFDBExtractorV2:
    LABEL_MAP = {"(N": 0, "(AFIB": 1, "(AFL": 2}

    def __init__(self, cfg: Config, rpeak_source: str):
        assert rpeak_source in ("annotation", "detector")
        self.cfg, self.source = cfg, rpeak_source
        self.record_stats = []

    def _records(self):
        recs = [l.strip() for l in open(Path(self.cfg.data_dir) / "RECORDS") if l.strip()]
        return [r for r in recs if r not in self.cfg.blacklist]

    def _annotation_peaks(self, path, rec):
        exts = ("qrsc",) if rec == "07859" else ("qrsc", "qrs")   # 07859 wajib .qrsc
        for ext in exts:
            if os.path.exists(f"{path}.{ext}"):
                return wfdb.rdann(path, ext).sample.astype(np.float64)
        return None

    def _peaks(self, path, rec, fs):
        """Return (peaks dalam sampel, fs dari peaks)."""
        if self.source == "annotation":
            return self._annotation_peaks(path, rec), float(fs)
        sig = np.nan_to_num(wfdb.rdrecord(path, channels=[self.cfg.ecg_channel]).p_signal[:, 0])
        x50 = hrv_core.resample_signal(sig, fs, hrv_core.FS_DEVICE)
        peaks = hrv_core.detect_rpeaks(x50, hrv_core.FS_DEVICE)
        # statistik kecocokan detektor vs anotasi (toleransi 100 ms)
        ref = self._annotation_peaks(path, rec)
        if ref is not None and len(peaks):
            p_orig = peaks * fs / hrv_core.FS_DEVICE
            tol = 0.1 * fs
            j = np.clip(np.searchsorted(ref, p_orig), 1, len(ref) - 1)
            d_det = np.minimum(np.abs(ref[j] - p_orig), np.abs(ref[j - 1] - p_orig))
            k = np.clip(np.searchsorted(p_orig, ref), 1, len(p_orig) - 1)
            d_ref = np.minimum(np.abs(p_orig[k] - ref), np.abs(p_orig[k - 1] - ref))
            self.record_stats.append({
                "record": rec, "n_anotasi": len(ref), "n_deteksi": len(peaks),
                "sensitivitas": np.mean(d_ref <= tol), "PPV": np.mean(d_det <= tol),
                "galat_median_ms": np.median(d_det[d_det <= tol]) / fs * 1000,
            })
        return peaks, float(hrv_core.FS_DEVICE)

    def process_record(self, rec):
        path = str(Path(self.cfg.data_dir) / rec)
        try:
            fs = wfdb.rdheader(path).fs
            atr = wfdb.rdann(path, "atr")
        except Exception as e:
            print(f"   [SKIP] {rec}: {e}")
            return []
        peaks, fs_pk = self._peaks(path, rec, fs)
        if peaks is None or len(peaks) < hrv_core.MIN_IBI + 1:
            print(f"   [SKIP] {rec}: R-peak tidak tersedia")
            return []

        # Anotasi ritme → label per IBI (ritme pada R-peak penutup IBI)
        rhythm = [(s, a.strip("\x00").strip()) for s, a in zip(atr.sample, atr.aux_note)
                  if a.strip("\x00").strip().startswith("(")]
        r_samp = np.array([s for s, _ in rhythm])
        r_lab = np.array([self.LABEL_MAP.get(a, -1) for _, a in rhythm])
        t_end, ibi = hrv_core.peaks_to_ibi(peaks, fs_pk)
        close = peaks[1:] * (fs / fs_pk)
        idx = np.searchsorted(r_samp, close, side="right") - 1
        lab = np.where(idx >= 0, r_lab[np.clip(idx, 0, None)], -1)

        rows, t, W = [], 0.0, hrv_core.WINDOW_SEC
        while t + W <= t_end[-1]:
            i0, i1 = np.searchsorted(t_end, [t, t + W], side="left")
            wl = lab[i0:i1]
            ok, _ = hrv_core.window_quality(ibi[i0:i1])
            if ok and wl.size and wl[0] >= 0 and np.all(wl == wl[0]):
                c = int(wl[0])
                rows.append((rec, t, ibi[i0:i1].copy(), c))
                t += self.cfg.stride_sec[c]
            else:
                t += self.cfg.invalid_step_sec
        return rows

    def run(self, use_cache=True):
        cache = Path(self.cfg.cache_dir) / f"dataset_v2_{self.source}.npz"
        stats_path = cache.with_suffix(".stats.csv")
        if use_cache and cache.exists():
            print(f"[cache] {cache}")
            z = np.load(cache)
            if stats_path.exists():
                self.record_stats = pd.read_csv(stats_path, dtype={"record": str}).to_dict("records")
            return {k: z[k] for k in z.files}

        rows = []
        for rec in self._records():
            t0 = time.perf_counter()
            r = self.process_record(rec)
            cnt = np.bincount([x[3] for x in r], minlength=3) if r else [0, 0, 0]
            print(f"  {self.source:<10} {rec}: {len(r):>6} window (N={cnt[0]}, AFIB={cnt[1]}, AFL={cnt[2]})"
                  f"  {time.perf_counter() - t0:.1f}s")
            rows.extend(r)

        seqs = [hrv_core.build_sequence(w[2]) for w in rows]
        data = {
            "X": np.stack([s[0] for s in seqs]).astype(np.float32),
            "M": np.stack([s[1] for s in seqs]).astype(np.float32),
            "F": np.stack([hrv_core.hrv_features(w[2]) for w in rows]).astype(np.float32),
            "y": np.array([w[3] for w in rows], dtype=np.int64),
            "records": np.array([w[0] for w in rows]),
            "t_start": np.array([w[1] for w in rows], dtype=np.float32),
        }
        np.savez_compressed(cache, **data)
        if self.record_stats:
            pd.DataFrame(self.record_stats).to_csv(stats_path, index=False)
        return data


ds, extractors = {}, {}
for src in ("annotation", "detector"):
    print(f"\n=== Ekstraksi: {src} ===")
    extractors[src] = AFDBExtractorV2(cfg, src)
    ds[src] = extractors[src].run()

summary = pd.DataFrame({
    src: {**{c: int((d["y"] == i).sum()) for i, c in enumerate(["Normal", "AFIB", "AFL"])},
          "total": len(d["y"]), "pasien": len(np.unique(d["records"]))}
    for src, d in ds.items()
}).T
display(summary)

### 3.1 Kualitas detektor R-peak 50 Hz terhadap anotasi

Sensitivitas dan PPV dihitung dengan toleransi 100 ms. Rekaman dengan nilai rendah adalah kandidat utama penyebab turunnya performa di produksi.

In [ ]:
det_stats = pd.DataFrame(extractors["detector"].record_stats)
if len(det_stats):
    display(det_stats.round(3).sort_values("sensitivitas"))
    print(f"Rata-rata: sensitivitas={det_stats['sensitivitas'].mean():.3f}, "
          f"PPV={det_stats['PPV'].mean():.3f}, galat median={det_stats['galat_median_ms'].median():.1f} ms")

## 4. Audit pasien & pembagian fold

Fold dibentuk **sekali di level pasien** (dari dataset `annotation`), lalu dipakai untuk kedua dataset. Dengan begitu perbandingan antar-sumber memakai pasien uji yang sama persis. Tabel di bawah menunjukkan berapa pasien AFL yang benar-benar ada di tiap fold.

In [ ]:
def make_patient_folds(d, cfg):
    sgkf = StratifiedGroupKFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.seed)
    rec2fold = {}
    for k, (_, va) in enumerate(sgkf.split(d["F"], d["y"], d["records"])):
        for r in np.unique(d["records"][va]):
            rec2fold[str(r)] = k
    return rec2fold


def fold_of(d):
    return np.array([REC2FOLD[str(r)] for r in d["records"]])


REC2FOLD = make_patient_folds(ds["annotation"], cfg)
FOLDS = {src: fold_of(d) for src, d in ds.items()}

A = ds["annotation"]
audit = []
for k in range(cfg.n_splits):
    sel = FOLDS["annotation"] == k
    recs = np.unique(A["records"][sel])
    afl_recs = np.unique(A["records"][sel & (A["y"] == 2)])
    audit.append({"fold": k, "pasien": len(recs), "pasien_AFL": len(afl_recs),
                  "rekaman_AFL": ", ".join(afl_recs),
                  **{c: int(((A["y"] == i) & sel).sum()) for i, c in enumerate(["N", "AFIB", "AFL"])}})
audit = pd.DataFrame(audit)
display(audit)

n_afl_pat = len(np.unique(A["records"][A["y"] == 2]))
print(f"Total pasien dengan AFL: {n_afl_pat}")
if (audit["pasien_AFL"] == 0).any():
    print("⚠ Ada fold tanpa pasien AFL → metrik AFL per fold tidak terdefinisi di fold itu. "
          "Karena itu metrik utama di bawah adalah pooled out-of-fold.")

## 5. Utilitas training & evaluasi

- Normalisasi z-score dan class weight dihitung **hanya dari train fold**.
- Setiap skenario menghasilkan prediksi **out-of-fold (OOF)** untuk seluruh sampel, sehingga metrik bisa dihitung secara pooled (lebih stabil untuk kelas yang sangat jarang) dan per pasien.
- `cv_*` menerima `test_ds` opsional: model dilatih pada satu sumber dan diuji pada sumber lain untuk pasien fold yang sama (eksperimen pergeseran domain).

In [ ]:
CLASSES = ["Normal", "AFIB", "AFL"]


def class_weights(y, K=3):
    counts = np.bincount(y, minlength=K).astype(np.float64)
    return len(y) / (K * np.maximum(counts, 1))


def zscore_fit(X, M):
    v = X[M.astype(bool)]
    return v.mean(0).astype(np.float32), (v.std(0) + 1e-8).astype(np.float32)


def zscore_apply(X, M, mu, sd):
    return ((X - mu) / sd * M[..., None]).astype(np.float32)


def train_dl(model_name, X, M, y, cfg, use_sampler=False, seed=0):
    set_seed(seed)
    mu, sd = zscore_fit(X, M)
    dset = TensorDataset(torch.from_numpy(zscore_apply(X, M, mu, sd)),
                         torch.from_numpy(M), torch.from_numpy(y))
    drop_last = len(y) > cfg.batch_size          # hindari batch berukuran 1 (BatchNorm)
    if use_sampler:
        sw = 1.0 / np.bincount(y, minlength=3)[y]
        sampler = WeightedRandomSampler(torch.as_tensor(sw, dtype=torch.float64), len(y), replacement=True)
        loader = DataLoader(dset, batch_size=cfg.batch_size, sampler=sampler, drop_last=drop_last)
        criterion = nn.CrossEntropyLoss()
    else:
        loader = DataLoader(dset, batch_size=cfg.batch_size, shuffle=True, drop_last=drop_last)
        criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights(y), dtype=torch.float32, device=DEVICE))

    net = build_model(model_name, **MODEL_KWARGS[model_name]).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)
    t0 = time.perf_counter()
    net.train()
    for _ in range(cfg.epochs):
        for bx, bm, by in loader:
            bx, bm, by = bx.to(DEVICE), bm.to(DEVICE), by.to(DEVICE)
            opt.zero_grad()
            criterion(net(bx, bm), by).backward()
            opt.step()
        sched.step()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    return net.eval(), mu, sd, time.perf_counter() - t0


@torch.no_grad()
def predict_dl(net, X, M, mu, sd, bs=4096):
    Xn = zscore_apply(X, M, mu, sd)
    out = []
    for s in range(0, len(Xn), bs):
        logits = net(torch.from_numpy(Xn[s:s + bs]).to(DEVICE), torch.from_numpy(M[s:s + bs]).to(DEVICE))
        out.append(torch.softmax(logits, 1).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, 3), np.float32)


def cv_dl(model_name, train_ds, folds_train, cfg, test_ds=None, folds_test=None, use_sampler=False):
    test_ds = train_ds if test_ds is None else test_ds
    folds_test = folds_train if folds_test is None else folds_test
    oof, times = np.full((len(test_ds["y"]), 3), np.nan, dtype=np.float32), []
    for k in range(cfg.n_splits):
        tr, te = folds_train != k, folds_test == k
        net, mu, sd, t = train_dl(model_name, train_ds["X"][tr], train_ds["M"][tr], train_ds["y"][tr],
                                  cfg, use_sampler=use_sampler, seed=cfg.seed + k)
        oof[te] = predict_dl(net, test_ds["X"][te], test_ds["M"][te], mu, sd)
        times.append(t)
        print(f"    fold {k}: {t:.1f}s")
    return oof, times


def make_lgbm(balanced=True):
    return lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31,
                              class_weight="balanced" if balanced else None,
                              random_state=cfg.seed, verbose=-1, n_jobs=-1)


def make_smote_pipe():
    # Scaler HANYA karena SMOTE berbasis jarak; LightGBM sendiri tidak butuh scaling.
    return ImbPipeline([("scaler", StandardScaler()),
                        ("smote", SMOTE(random_state=cfg.seed, k_neighbors=5)),
                        ("model", make_lgbm(balanced=False))])


def cv_tabular(make_model, train_ds, folds_train, cfg, test_ds=None, folds_test=None):
    test_ds = train_ds if test_ds is None else test_ds
    folds_test = folds_train if folds_test is None else folds_test
    oof, times = np.full((len(test_ds["y"]), 3), np.nan, dtype=np.float32), []
    for k in range(cfg.n_splits):
        tr, te = folds_train != k, folds_test == k
        model = make_model()
        t0 = time.perf_counter()
        model.fit(train_ds["F"][tr], train_ds["y"][tr])
        times.append(time.perf_counter() - t0)
        if te.any():
            oof[te] = model.predict_proba(test_ds["F"][te])
    return oof, times


def evaluate(name, y, proba, folds, records):
    pred = proba.argmax(1)
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=[0, 1, 2], zero_division=0)
    per_fold = [f1_score(y[folds == k], pred[folds == k], labels=np.unique(y[folds == k]),
                         average="macro", zero_division=0) for k in np.unique(folds)]
    afl_recs = np.unique(records[y == 2])
    afl_pat = {str(rc): float(np.mean(pred[(records == rc) & (y == 2)] == 2)) for rc in afl_recs}
    return {
        "Model": name,
        "Macro-F1 (pooled)": f1_score(y, pred, labels=[0, 1, 2], average="macro", zero_division=0),
        "Macro-F1 per fold": f"{np.mean(per_fold):.3f} ± {np.std(per_fold):.3f}",
        "Normal R": r[0], "AFIB P": p[1], "AFIB R": r[1],
        "AFL P": p[2], "AFL R": r[2], "AFL F1": f[2],
        "AFL R per pasien (min–maks)": (f"{min(afl_pat.values()):.2f}–{max(afl_pat.values()):.2f}"
                                       if afl_pat else "-"),
        "_cm": confusion_matrix(y, pred, labels=[0, 1, 2], normalize="true"),
        "_afl_per_patient": afl_pat,
    }


def show_results(rows):
    df = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in rows])
    return df.round(3)


def plot_cms(rows, title=""):
    n = len(rows)
    fig, axes = plt.subplots(1, n, figsize=(4.2 * n, 3.8), squeeze=False)
    for ax, r in zip(axes[0], rows):
        sns.heatmap(r["_cm"], annot=True, fmt=".2f", cmap="Blues", cbar=False, ax=ax,
                    xticklabels=CLASSES, yticklabels=CLASSES, vmin=0, vmax=1)
        ax.set_title(r["Model"], fontsize=10); ax.set_xlabel("Prediksi"); ax.set_ylabel("Aktual")
    fig.suptitle(title); plt.tight_layout(); plt.show()


# ── Kalibrasi bias kelas (pengganti argmax polos) ──
def _macro_f1_fast(y, pred, K=3):
    cm = np.bincount(y * K + pred, minlength=K * K).reshape(K, K)
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    denom = 2 * tp + fp + fn
    f1 = np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)
    return f1[cm.sum(1) > 0].mean()


def fit_class_bias(y, proba, grid=np.linspace(-3, 3, 25)):
    logp = np.log(np.clip(proba, 1e-9, 1))
    best, best_b = -1, np.zeros(3)
    for b1 in grid:
        for b2 in grid:
            b = np.array([0.0, b1, b2])
            s = _macro_f1_fast(y, (logp + b).argmax(1))
            if s > best:
                best, best_b = s, b
    return best_b


def apply_bias(proba, b):
    z = np.log(np.clip(proba, 1e-9, 1)) + b
    z = np.exp(z - z.max(1, keepdims=True))
    return z / z.sum(1, keepdims=True)


def crossfit_bias(y, proba, folds):
    """Bias fold k dipelajari dari OOF fold lain → evaluasi tetap jujur."""
    adj = np.empty_like(proba)
    for k in np.unique(folds):
        b = fit_class_bias(y[folds != k], proba[folds != k])
        adj[folds == k] = apply_bias(proba[folds == k], b)
    return adj

## 6. Benchmark utama (sumber `annotation`, sebanding dengan notebook lama)

Skenario A memakai 16 fitur HRV. Skenario B memakai sekuens `[IBI, ΔIBI]`. Semua dievaluasi dengan fold pasien yang sama.

Sebagai pembanding, hasil notebook lama (rata-rata per fold, stride AFL 1 s): Micro-Transformer B1 Macro-F1 0,826, AFL recall 0,76; LightGBM A1 Macro-F1 0,678. Angka di bawah **tidak 1:1 sebanding** karena stride, fitur, dan cara agregasinya berbeda; fokuskan pada perbandingan antar-model di dalam notebook ini.

In [ ]:
A, fA = ds["annotation"], FOLDS["annotation"]
RESULTS, OOF, TRAIN_TIME = [], {}, {}


def register(name, d, folds, oof, times):
    RESULTS.append(evaluate(name, d["y"], oof, folds, d["records"]))
    OOF[name], TRAIN_TIME[name] = oof, float(np.mean(times))


print("A1 LightGBM + class_weight (16 fitur HRV)")
register("A1 LightGBM", A, fA, *cv_tabular(make_lgbm, A, fA, cfg))

print("A2 LightGBM + SMOTE in-fold")
register("A2 LightGBM+SMOTE", A, fA, *cv_tabular(make_smote_pipe, A, fA, cfg))

for m in cfg.dl_models:
    print(f"B1 {m} + weighted loss")
    register(f"B1 {m}", A, fA, *cv_dl(m, A, fA, cfg))

print("B2 MicroTransformer + WeightedRandomSampler")
register("B2 MicroTransformer", A, fA, *cv_dl("MicroTransformer", A, fA, cfg, use_sampler=True))

res_main = show_results(RESULTS)
res_main["Train/fold (s)"] = [round(TRAIN_TIME[r["Model"]], 1) for r in RESULTS]
display(res_main)
plot_cms(RESULTS, "Confusion matrix pooled OOF (normalisasi per baris = recall)")

In [ ]:
# Recall AFL per pasien untuk tiap model: memperlihatkan seberapa bergantung hasil pada 1–2 pasien
afl_tbl = pd.DataFrame({r["Model"]: r["_afl_per_patient"] for r in RESULTS})
afl_tbl.index.name = "rekaman AFL"
afl_tbl["jumlah window AFL"] = [int(((A["records"] == rc) & (A["y"] == 2)).sum()) for rc in afl_tbl.index]
display(afl_tbl.round(2))

## 7. Kalibrasi bias kelas

Alih-alih `argmax` polos, ditambahkan offset log-probabilitas per kelas yang memaksimalkan Macro-F1. Agar jujur, bias untuk fold *k* dipelajari dari OOF fold lain (*cross-fitting*), sehingga angka "sesudah" tidak bocor.

In [ ]:
calib_rows = []
for r in RESULTS:
    name = r["Model"]
    adj = crossfit_bias(A["y"], OOF[name], fA)
    after = evaluate(name, A["y"], adj, fA, A["records"])
    calib_rows.append({"Model": name,
                       "Macro-F1 sebelum": r["Macro-F1 (pooled)"], "Macro-F1 sesudah": after["Macro-F1 (pooled)"],
                       "AFL P sebelum": r["AFL P"], "AFL P sesudah": after["AFL P"],
                       "AFL R sebelum": r["AFL R"], "AFL R sesudah": after["AFL R"]})
display(pd.DataFrame(calib_rows).round(3))

## 8. Eksperimen pergeseran domain (temuan paling penting)

Pertanyaan: seberapa jauh performa turun saat input berasal dari detektor 50 Hz, seperti di perangkat?

| Skenario | Latih pada | Uji pada | Makna |
|---|---|---|---|
| S1 | anotasi 250 Hz | anotasi 250 Hz | kondisi notebook lama (optimis) |
| S2 | anotasi 250 Hz | detektor 50 Hz | **apa yang terjadi pada `final.joblib` lama di produksi** |
| S3 | detektor 50 Hz | detektor 50 Hz | latih sesuai kondisi produksi |

Semua skenario memakai pasien uji yang sama per fold.

In [ ]:
D, fD = ds["detector"], FOLDS["detector"]
SHIFT = []
for label, runner in [
    ("LightGBM", lambda tr, ftr, te, fte: cv_tabular(make_lgbm, tr, ftr, cfg, te, fte)),
    ("MicroTransformer", lambda tr, ftr, te, fte: cv_dl("MicroTransformer", tr, ftr, cfg, te, fte)),
]:
    s1 = OOF["A1 LightGBM" if label == "LightGBM" else "B1 MicroTransformer"]
    print(f"{label}: S2 latih anotasi → uji detektor"); s2, _ = runner(A, fA, D, fD)
    print(f"{label}: S3 latih detektor → uji detektor"); s3, t3 = runner(D, fD, D, fD)
    OOF[f"S3 {label}"] = s3
    for tag, d, f, p in [("S1 anot→anot", A, fA, s1), ("S2 anot→det", D, fD, s2), ("S3 det→det", D, fD, s3)]:
        SHIFT.append(evaluate(f"{label} | {tag}", d["y"], p, f, d["records"]))

display(show_results(SHIFT))
plot_cms(SHIFT[3:], "MicroTransformer: S1 / S2 / S3")

**Cara membaca.** Jika S2 jauh di bawah S1, model lama akan bekerja lebih buruk di perangkat daripada yang dilaporkan. Jika S3 memulihkan sebagian besar selisih itu, model akhir sebaiknya dilatih pada data hasil detektor (`cfg.final_source = "detector"`, default). Jika S3 pun tetap rendah untuk AFL, informasi RR dari sinyal 50 Hz memang belum cukup, dan langkah berikutnya adalah menambah input sinyal EKG atau data AFL dari dataset lain.

## 9. Latensi yang adil

Semua model diukur di **CPU, 1 thread**, dengan `time.perf_counter`, termasuk praproses dari IBI mentah (fitur HRV untuk LightGBM, pembentukan sekuens + normalisasi untuk DL). Ada dua ukuran: 1 window (kasus real-time) dan batch 1.000 window. Latensi tidak bergantung pada nilai bobot, jadi model DL tidak perlu dilatih ulang di sini.

In [ ]:
def bench(fn, n=200, warmup=10):
    for _ in range(warmup):
        fn()
    ts = []
    for _ in range(n):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts) * 1000)


prev_threads = torch.get_num_threads()
torch.set_num_threads(1)
ibi_list = [A["X"][i, :, 0][A["M"][i] > 0] for i in range(min(1000, len(A["y"])))]
one = ibi_list[0]
mu_a, sd_a = zscore_fit(A["X"], A["M"])

lgbm_full = make_lgbm(); lgbm_full.set_params(n_jobs=1); lgbm_full.fit(A["F"], A["y"])
lat = [{
    "Model": "LightGBM (16 fitur)",
    "1 window (ms)": bench(lambda: lgbm_full.predict_proba(hrv_core.hrv_features(one)[None])),
    "1.000 window (ms)": bench(lambda: lgbm_full.predict_proba(np.stack([hrv_core.hrv_features(b) for b in ibi_list])), n=5, warmup=1),
}]
for m in cfg.dl_models:
    net = build_model(m, **MODEL_KWARGS[m]).eval()

    @torch.no_grad()
    def run(batch, net=net):
        seqs = [hrv_core.build_sequence(b) for b in batch]
        X = np.stack([s[0] for s in seqs]); M = np.stack([s[1] for s in seqs])
        return net(torch.from_numpy(zscore_apply(X, M, mu_a, sd_a)), torch.from_numpy(M))

    lat.append({"Model": f"{m} ({count_params(net):,} param)",
                "1 window (ms)": bench(lambda: run([one])),
                "1.000 window (ms)": bench(lambda: run(ibi_list), n=10, warmup=2)})
torch.set_num_threads(prev_threads)
display(pd.DataFrame(lat).round(2))

## 10. Training final & ekspor

Model `cfg.final_model` dilatih ulang pada **seluruh** data sumber `cfg.final_source`, dengan jumlah epoch yang sama seperti di CV (tanpa early stopping, jadi tidak ada data validasi yang ikut memilih model). Bias kelas diambil dari OOF CV pada sumber yang sama.

Ekspor:
- `model_export/sihedaf_v2.pt` — hanya `state_dict` (tanpa pickle objek).
- `model_export/sihedaf_v2.json` — arsitektur, `mu`/`sigma`, bias kelas, konstanta praproses, ringkasan data latih, metrik CV, versi pustaka, commit git, checksum bobot.

In [ ]:
FD = ds[cfg.final_source]
f_final = FOLDS[cfg.final_source]

# OOF pada sumber final untuk bias & metrik yang dilaporkan
oof_key = {("detector", "MicroTransformer"): "S3 MicroTransformer",
           ("annotation", cfg.final_model): f"B1 {cfg.final_model}"}.get((cfg.final_source, cfg.final_model))
if oof_key in OOF:
    oof_final = OOF[oof_key]
else:
    print("Menjalankan CV untuk konfigurasi final...")
    oof_final, _ = cv_dl(cfg.final_model, FD, f_final, cfg)

class_bias = fit_class_bias(FD["y"], oof_final)
cv_raw = evaluate("raw", FD["y"], oof_final, f_final, FD["records"])
cv_cal = evaluate("calibrated", FD["y"], crossfit_bias(FD["y"], oof_final, f_final), f_final, FD["records"])
print("Bias kelas (log-prob):", class_bias.round(2))

print(f"Melatih {cfg.final_model} final pada {len(FD['y']):,} window ({cfg.final_source})...")
final_net, final_mu, final_sd, final_time = train_dl(cfg.final_model, FD["X"], FD["M"], FD["y"], cfg, seed=cfg.seed)

out = Path(cfg.out_dir); name = "sihedaf_v2"
weights_path = out / f"{name}.pt"
torch.save({k: v.detach().cpu() for k, v in final_net.state_dict().items()}, weights_path)


def _git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return None


def _metrics(r):
    return {k: (float(v) if isinstance(v, (float, np.floating)) else v)
            for k, v in r.items() if not k.startswith("_") and k != "Model"}


meta = {
    "model_name": cfg.final_model,
    "model_config": MODEL_KWARGS[cfg.final_model],
    "labels_map": {str(k): v for k, v in hrv_core.LABELS_MAP.items()},
    "mu": final_mu.tolist(), "sigma": final_sd.tolist(),
    "class_bias": class_bias.tolist(),
    "preprocessing": hrv_core.preprocessing_config(),
    "training": {
        "rpeak_source": cfg.final_source, "epochs": cfg.epochs, "lr": cfg.lr,
        "weight_decay": cfg.weight_decay, "batch_size": cfg.batch_size, "loss": "weighted cross-entropy",
        "stride_sec": {str(k): v for k, v in cfg.stride_sec.items()},
        "n_windows": int(len(FD["y"])), "class_counts": np.bincount(FD["y"], minlength=3).tolist(),
        "records": sorted(map(str, np.unique(FD["records"]))), "train_time_sec": round(final_time, 1),
    },
    "cv_metrics": {"argmax": _metrics(cv_raw), "dengan_bias_crossfit": _metrics(cv_cal)},
    "versions": {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__},
    "git_commit": _git_commit(),
    "weights_sha256": hashlib.sha256(weights_path.read_bytes()).hexdigest(),
    "created": time.strftime("%Y-%m-%d %H:%M:%S"),
}
(out / f"{name}.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))
print(f"✓ Tersimpan: {weights_path} dan {out / (name + '.json')}")
print(json.dumps(meta["cv_metrics"], indent=2, ensure_ascii=False))

## 11. Predictor produksi

Ditulis ke `predictor.py`. Tiga pintu masuk: `predict_proba_sequences` (sekuens siap pakai), `predict_ibi` (perangkat mengirim IBI dalam ms), dan `predict_ecg` (perangkat mengirim sinyal EKG mentah, fs berapa pun).

In [ ]:
%%writefile predictor.py
"""
predictor.py — inferensi produksi SIHEDAF v2.

Perbedaan dari versi lama:
- Memuat bobot dari .pt (state_dict, torch.load weights_only=True) + metadata
  .json. Tidak ada pickle/joblib → aman dimuat dan tidak terikat versi pickle.
- Verifikasi checksum SHA-256 bobot terhadap metadata.
- Praproses memakai hrv_core (kode yang sama dengan training).
- TIDAK ADA fallback diam-diam: file hilang / sinyal buruk → error atau
  status eksplisit, bukan prediksi dari bobot acak / scaler dummy.
- Bias per kelas hasil kalibrasi CV ikut diterapkan.
"""
import hashlib
import json
from pathlib import Path
from typing import Optional

import numpy as np
import torch

import hrv_core
from sihedaf_models import build_model


class ECGPredictor:
    def __init__(self, model_dir: str = "model_export", name: str = "sihedaf_v2",
                 device: Optional[str] = None):
        model_dir = Path(model_dir)
        meta_path = model_dir / f"{name}.json"
        weight_path = model_dir / f"{name}.pt"
        if not meta_path.exists() or not weight_path.exists():
            raise FileNotFoundError(f"Butuh {meta_path} dan {weight_path}")

        self.meta = json.loads(meta_path.read_text())
        digest = hashlib.sha256(weight_path.read_bytes()).hexdigest()
        if digest != self.meta["weights_sha256"]:
            raise ValueError("Checksum bobot tidak cocok dengan metadata — file tertukar/rusak.")

        pre = self.meta["preprocessing"]
        if pre != hrv_core.preprocessing_config():
            raise ValueError(
                "Konstanta hrv_core berbeda dari saat training. "
                f"Training: {pre} | Sekarang: {hrv_core.preprocessing_config()}"
            )

        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.mu = np.asarray(self.meta["mu"], dtype=np.float32)
        self.sigma = np.asarray(self.meta["sigma"], dtype=np.float32)
        self.class_bias = np.asarray(self.meta.get("class_bias", [0.0, 0.0, 0.0]), dtype=np.float32)
        self.labels_map = {int(k): v for k, v in self.meta["labels_map"].items()}

        self.model = build_model(self.meta["model_name"], **self.meta["model_config"]).to(self.device)
        state = torch.load(weight_path, map_location=self.device, weights_only=True)
        self.model.load_state_dict(state)
        self.model.eval()

    # ── level sekuens ──────────────────────────────────────────────────
    @torch.no_grad()
    def predict_proba_sequences(self, X: np.ndarray, mask: np.ndarray) -> np.ndarray:
        """X: (N, MAX_BEATS, 2) dalam ms, mask: (N, MAX_BEATS) → prob (N, 3)."""
        X = np.asarray(X, dtype=np.float32)
        mask = np.asarray(mask, dtype=np.float32)
        if X.ndim == 2:
            X, mask = X[None], mask[None]
        Xn = (X - self.mu) / self.sigma * mask[..., None]
        logits = self.model(torch.from_numpy(Xn).to(self.device),
                            torch.from_numpy(mask).to(self.device)).cpu().numpy()
        logp = logits - np.logaddexp.reduce(logits, axis=1, keepdims=True)
        adj = logp + self.class_bias
        adj = np.exp(adj - adj.max(axis=1, keepdims=True))
        return adj / adj.sum(axis=1, keepdims=True)

    def _format(self, prob: np.ndarray) -> dict:
        cid = int(np.argmax(prob))
        return {
            "status": "ok",
            "class_id": cid,
            "class_name": self.labels_map[cid],
            "confidence": float(prob[cid]),
            "probabilities": {self.labels_map[i]: float(p) for i, p in enumerate(prob)},
        }

    # ── level IBI (jika perangkat sudah mengirim IBI) ──────────────────
    def predict_ibi(self, ibi_ms) -> dict:
        ibi_ms = np.asarray(ibi_ms, dtype=np.float64)
        if np.nanmedian(ibi_ms) < 10:
            raise ValueError("IBI tampak dalam DETIK. Kirim dalam MILIDETIK (mis. 800, bukan 0.8).")
        ok, reason = hrv_core.window_quality(ibi_ms)
        if not ok:
            return {"status": "tidak_terbaca", "reason": reason}
        X, m = hrv_core.build_sequence(ibi_ms)
        return self._format(self.predict_proba_sequences(X, m)[0])

    # ── level sinyal EKG mentah ────────────────────────────────────────
    def predict_ecg(self, ecg, fs: float, stride_sec: float = 5.0) -> dict:
        """
        EKG mentah → prediksi per window 10 detik + ringkasan (rata-rata
        probabilitas window yang valid). fs = sampling rate ASLI sinyal.
        """
        windows, peaks = hrv_core.signal_to_windows(np.asarray(ecg), fs, stride_sec)
        valid = [w for w in windows if w["ok"]]
        per_window = []
        if valid:
            seqs = [hrv_core.build_sequence(w["ibi_ms"]) for w in valid]
            X = np.stack([s[0] for s in seqs]); M = np.stack([s[1] for s in seqs])
            probs = self.predict_proba_sequences(X, M)
            for w, p in zip(valid, probs):
                r = self._format(p); r["start_sec"] = w["start_sec"]
                per_window.append(r)
        for w in windows:
            if not w["ok"]:
                per_window.append({"status": "tidak_terbaca", "start_sec": w["start_sec"], "reason": w["reason"]})
        per_window.sort(key=lambda r: r["start_sec"])

        dur = len(ecg) / fs
        hr = 60.0 * (len(peaks) - 1) / ((peaks[-1] - peaks[0]) / hrv_core.FS_DEVICE) if len(peaks) > 2 else None
        summary = {"status": "tidak_terbaca", "reason": "tidak ada window valid"}
        if valid:
            summary = self._format(np.mean(probs, axis=0))
        summary.update({
            "duration_sec": round(dur, 1), "n_rpeaks": int(len(peaks)),
            "heart_rate_bpm": round(hr, 1) if hr else None,
            "n_windows": len(windows), "n_windows_valid": len(valid),
        })
        return {"summary": summary, "windows": per_window}

In [ ]:
import predictor
importlib.reload(predictor)
P = predictor.ECGPredictor(cfg.out_dir)

# ── Uji paritas: predictor == model di notebook (termasuk bias) ──
rng = np.random.default_rng(0)
idx = rng.choice(len(FD["y"]), size=min(500, len(FD["y"])), replace=False)
p_nb = apply_bias(predict_dl(final_net, FD["X"][idx], FD["M"][idx], final_mu, final_sd), class_bias)
p_api = P.predict_proba_sequences(FD["X"][idx], FD["M"][idx])
diff = np.abs(p_nb - p_api).max()
print(f"Selisih maks notebook vs predictor: {diff:.2e}")
assert diff < 1e-4

# ── Uji guard: IBI dalam detik harus ditolak, bukan diprediksi diam-diam ──
try:
    P.predict_ibi([0.8, 0.82, 0.79, 0.81, 0.8])
    raise AssertionError("IBI dalam detik seharusnya ditolak")
except ValueError as e:
    print("✓ Guard satuan bekerja:", e)

print(P.predict_ibi([800, 810, 790, 805, 795, 800, 812, 798, 803, 801]))
print(P.predict_ibi([800, 900]))   # terlalu sedikit detak → status tidak_terbaca

In [ ]:
# ── Uji end-to-end: potongan 60 detik EKG AFDB (250 Hz) → predict_ecg ──
def first_segment_of_class(d, cls):
    i = int(np.where(d["y"] == cls)[0][0]) if (d["y"] == cls).any() else None
    return None if i is None else (str(d["records"][i]), float(d["t_start"][i]))

rows = []
for cls in (0, 1, 2):
    seg = first_segment_of_class(ds["annotation"], cls)
    if seg is None:
        continue
    rec, t0 = seg
    path = str(Path(cfg.data_dir) / rec)
    fs = wfdb.rdheader(path).fs
    sig = wfdb.rdrecord(path, channels=[cfg.ecg_channel], sampfrom=int(t0 * fs),
                        sampto=int((t0 + 60) * fs)).p_signal[:, 0]
    s = P.predict_ecg(sig, fs)["summary"]
    rows.append({"label asli": CLASSES[cls], "rekaman": rec, "mulai (s)": t0,
                 "prediksi": s.get("class_name"), "confidence": s.get("confidence"),
                 "HR (bpm)": s["heart_rate_bpm"], "window valid": f"{s['n_windows_valid']}/{s['n_windows']}"})
display(pd.DataFrame(rows))
print("Catatan: rekaman ini ikut data latih model final, jadi ini uji alur, bukan uji akurasi.")

In [ ]:
# ── Uji data perangkat (raw_71.json, 50 Hz), jika file tersedia ──
raw_path = Path(cfg.raw_test_file)
if raw_path.exists():
    raw = np.asarray(json.loads(raw_path.read_text()), dtype=float).ravel()
    result = P.predict_ecg(raw, fs=cfg.raw_test_fs)
    print(json.dumps(result["summary"], indent=2, ensure_ascii=False))
    display(pd.DataFrame(result["windows"]).drop(columns=["probabilities"], errors="ignore"))

    _, peaks = hrv_core.signal_to_windows(raw, cfg.raw_test_fs)
    x50 = hrv_core.resample_signal(raw, cfg.raw_test_fs)
    t = np.arange(x50.size) / hrv_core.FS_DEVICE
    plt.figure(figsize=(14, 3))
    plt.plot(t, x50, lw=0.7)
    pk = np.clip(np.round(peaks).astype(int), 0, x50.size - 1)
    plt.plot(pk / hrv_core.FS_DEVICE, x50[pk], "rx")
    plt.title("Deteksi R-peak pada data perangkat — periksa secara visual"); plt.xlabel("detik")
    plt.tight_layout(); plt.show()
else:
    print(f"{raw_path} tidak ditemukan — lewati uji data perangkat.")

## 12. File pendukung repo

In [ ]:
%%writefile requirements.txt
numpy>=1.26
scipy>=1.11
pandas>=2.0
scikit-learn>=1.4
imbalanced-learn>=0.12
lightgbm>=4.0
torch>=2.1
wfdb>=4.1
matplotlib>=3.8
seaborn>=0.13

## 13. Ringkasan & langkah berikutnya

File yang dihasilkan notebook ini dan perlu di-commit: `hrv_core.py`, `sihedaf_models.py`, `predictor.py`, `requirements.txt`, `model_export/sihedaf_v2.pt`, `model_export/sihedaf_v2.json`, dan notebook ini.

File lama yang bisa dihapus setelah backend pindah: `final.joblib`, `predictor.py` lama di folder `cek lama/`, dan `cek lama/dokumentasi.md`. Tambahkan `cache/` ke `.gitignore`.

Untuk backend:

```python
from predictor import ECGPredictor
P = ECGPredictor("model_export")
hasil = P.predict_ecg(sinyal_mentah, fs=50)      # atau P.predict_ibi(ibi_ms)
```

Hal yang **masih perlu** dilakukan di luar notebook ini:

1. Validasi pada rekaman dari perangkat asli yang dilabeli dokter. AFDB yang di-downsample hanya mensimulasikan 50 Hz; noise sensor, posisi elektroda, dan gerak pasien belum ikut tersimulasi.
2. Jika S3 masih lemah untuk AFL, tambahkan data AFL dari dataset lain (mis. PTB-XL, Chapman-Shaoxing) atau cabang input sinyal EKG.
3. Bandingkan dengan model pra-latih (ECGFounder, MOMENT) memakai fold pasien yang sama (`REC2FOLD`).